# Retreiver Demonstration

In [3]:
# Importing packages
import pandas as pd
import torch
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, logging, AutoModel
logging.set_verbosity_error()
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
import os
from torch import Tensor
import faiss 
import json
from beir.datasets.data_loader import GenericDataLoader
import torch.nn.functional as F

In [4]:
# Loading token
load_dotenv('token.env')
token = os.getenv('HUGGINGFACE_TOKEN')

# Loading model - pass token directly
model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    token=token  # Pass token here
)

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct.
401 Client Error. (Request ID: Root=1-69790c91-6e0f92bd31f156fc730de100;380d1d06-b5b8-4831-a934-42a74ce533f0)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-8B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
# Loading Training Data:
data_dir = "/work/mbouthil/datasets/msmarco"
corpus, queries, qrels = GenericDataLoader(data_folder=data_dir).load(split="train")

100%|██████████| 8841823/8841823 [01:16<00:00, 115315.01it/s]


In [8]:
# Loading Test Data:
test_corpus, test_queries, test_qrels = GenericDataLoader(data_folder=data_dir).load(split="test")

100%|██████████| 8841823/8841823 [01:22<00:00, 107208.67it/s]


# Data Exploration

### Query Data Exploration

In [27]:
n = 10
i = 0

print('-Query ID--------Query Text-')
for key, value in queries.items():
    print('{'+ key, ":", value, "}")
    i += 1
    if i == n:
        break

-Query ID--------Query Text-
{1185869 : )what was the immediate impact of the success of the manhattan project? }
{1185868 : _________ justice is designed to repair the harm to victim, the community and the offender caused by the offender criminal act. question 19 options: }
{597651 : what color is amber urine }
{403613 : is autoimmune hepatitis a bile acid synthesis disorder }
{1183785 : elegxo meaning }
{312651 : how much does an average person make for tutoring }
{80385 : can you use a calculator on the compass test }
{645590 : what does physical medicine do }
{645337 : what does pending mean on listing }
{186154 : feeding rice cereal how many times per day }


In [47]:
print(f"There are a total of {len(queries):,} training queries")
print(f"There are a total of {len(test_queries):,} testing queries")
print("\n")

mean_length = np.mean([len(queries[key]) for key in queries.keys()])
print(f"The queries have a mean character length of {mean_length:.2f}")
mean_length = np.mean([len(queries[key].split()) for key in queries.keys()])
print(f"The queries have a mean word count of {mean_length:.2f}")

There are a total of 502,939 training queries
There are a total of 43 testing queries


The queries have a mean character length of 33.22
The queries have a mean word count of 5.97


### Passage Data Exporation

In [28]:
n = 10
i = 0
print('-Passage ID--------Passage Text-')
for key, value in corpus.items():
    print('{' + key, ":", value['text'], "}")
    i += 1
    if i == n:
        break

-Passage ID--------Passage Text-
{0 : The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated. }
{1 : The Manhattan Project and its atomic bomb helped bring an end to World War II. Its legacy of peaceful uses of atomic energy continues to have an impact on history and science. }
{2 : Essay on The Manhattan Project - The Manhattan Project The Manhattan Project was to see if making an atomic bomb possible. The success of this project would forever change the world forever making it known that something this powerful can be manmade. }
{3 : The Manhattan Project was the name for a project conducted during World War II, to develop the first atomic bomb. It refers specifically to the period of the project from 194 â¦ 2-1946 un

In [48]:
print(f"There are a total of {len(corpus):,} training passages")
print(f"There are a total of {len(test_corpus):,} testing passages")
print("\n")

mean_length = np.mean([len(corpus[key]['text']) for key in corpus.keys()])
print("The corpus (pasages) has a mean character length of {mean_length:.2f}")
mean_length = np.mean([len(corpus[key]['text'].split()) for key in corpus.keys()])
print("The corpus (pasages) has a mean word count of {mean_length:.2f}")

There are a total of 8,841,823 training passages
There are a total of 8,841,823 testing passages


The corpus (pasages) has a mean character length of {mean_length:.2f}
The corpus (pasages) has a mean word count of {mean_length:.2f}


### Qrels Data Exploration

In [32]:
n = 10
i = 0

print('-Query ID--------dict(passage ID: Score)-')
for key, value in qrels.items():
    print('{'+ key, ":", value, "}")
    i += 1
    if i == n:
        break

-Query ID--------dict(passage ID: Score)-
{1185869 : {'0': 1} }
{1185868 : {'16': 1} }
{597651 : {'49': 1} }
{403613 : {'60': 1} }
{1183785 : {'389': 1} }
{312651 : {'616': 1} }
{80385 : {'723': 1} }
{645590 : {'944': 1} }
{645337 : {'1054': 1} }
{186154 : {'1160': 1} }


In [42]:
print(len(qrels))
print(len(qrels) ==  len(queries))
print("")
print("The qrels provide the mapping of queries to relevant passage(s)")

502939
True

The qrels provide the mapping of queries to relevant passage(s)


In [43]:
# n = 200
# i = 0
# print("")
# for key, value in qrels.items():
#     if len(value) > 1:
#         print('{', key, ":", value, "}")
#     i += 1
#     if i == n:
#         break

# Retreiver Demonstration

### We begin by loading in our Vector Database:

**Recall:** this vector database is constructed using the trained Passage Encoder

In [49]:
# Loading Index
index = faiss.read_index("/work/mbouthil/projects/research_project/RAG/retrieval_data/passage.index")

# Load Metadata
metadata = []
with open("/work/mbouthil/projects/research_project/RAG/retrieval_data/passage_metadata.jsonl") as f:
    for line in f:
        metadata.append(json.loads(line))

### Load the trained Query Encoder

In [50]:
# Loading Query Encoder
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
query_encoder = AutoModel.from_pretrained(
    "/work/mbouthil/projects/research_project/RAG/model_weights/query_encoder"
) # .to("cuda")

query_encoder.eval()

def encode_query(query:str, batch_size:int=32) -> Tensor:

    embeddings = []

    with torch.no_grad():
        inputs = tokenizer(
            query, 
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=32
        ) #.to("cuda")

    emb = query_encoder(**inputs).last_hidden_state[:, 0]
    emb = F.normalize(emb, p=2, dim=-1)

    embeddings.append(emb.cpu())

    return torch.cat(embeddings, 0)

### Embedding the Quesetion

In [ ]:
question = queries['8']
print(question)

# Embedding Query
q_emb = encode_query([question]).detach().cpu().numpy()
print('Embedding length: ', q_emb.shape[1])

 In humans, the normal set point for body temperature is 
Embedding shape:  768


In [57]:
# Matching for the top K=10 highest scores


index = faiss.IndexFlatIP(q_emb.shape[1])
K = 10
scores, ids = index.search(q_emb, K)
scores = scores[0]
ids = ids[0]


# scores = list(scores[0]).re
# versal()
# ids = list(ids[0]).reversal()
# candidates = [metadata[i] for i in ids[0]]
# test_df = pd.DataFrame({'scores': scores, 'passage':candidates})
# test_df.head(10)

In [12]:
scores = scores[0]

ids = ids[0]

In [15]:
print('There is an issue with the FAISS database saving')

There is an issue with the FAISS database saving


In [55]:
for i, id in enumerate(ids):
    print(scores[i])
    print(corpus[str(id)]['text'])
    print("\n\n")

-3.4028235e+38


KeyError: '-1'

# Syn Q MARCO

In [16]:
data_dir = "/work/mbouthil/projects/research_project/RAG/datasets/msmarco_modified"
corpus, s_queries, s_qrels = GenericDataLoader(data_folder=data_dir).load(split="train")

  0%|          | 0/8841823 [00:00<?, ?it/s]

In [17]:
len(s_queries)

602939

In [18]:
print("Total original queries:", len(queries))
print("Total original + synthetic:", len(s_queries))

Total original queries: 502939
Total original + synthetic: 602939


In [19]:
max_id = max([int(id) for id in queries.keys()]) + 1
keys = list(queries.keys())
n = 10


for i in range(n):
    print('----------------------------------------------------------------------------------------------')
    print('**Original Query:** \n \t', queries[keys[i]])
    print('\n')
    print('**New Synthetic Query created using zero shot prompting LLM:**\n\t', s_queries[str(max_id + i)])
    print('----------------------------------------------------------------------------------------------')
    print('\n\n')

----------------------------------------------------------------------------------------------
**Original Query:** 
 	 )what was the immediate impact of the success of the manhattan project?


**New Synthetic Query created using zero shot prompting LLM:**
	 What were the direct consequences of the Manhattan Project's achievement?
----------------------------------------------------------------------------------------------



----------------------------------------------------------------------------------------------
**Original Query:** 
 	 _________ justice is designed to repair the harm to victim, the community and the offender caused by the offender criminal act. question 19 options:


**New Synthetic Query created using zero shot prompting LLM:**
	 What is the primary goal of restorative justice in addressing the consequences of a criminal offense?
----------------------------------------------------------------------------------------------



-----------------------------------

These use the same qrels mapping the original queries to the passages. Hence, no additional passages are require to be created, and the qrels mapping are simply copied. I.e:

In [21]:
print('Original Qrels mapping the query to the positive passage', qrels[keys[0]])
print('Syntehtic Qrels Mapping the Synthtic query to the positive passage:', s_qrels[str(max_id)])
print('\n')
print(s_qrels[str(max_id)] == qrels[keys[0]])

Original Qrels mapping the query to the positive passage {'0': 1}
Syntehtic Qrels Mapping the Synthtic query to the positive passage: {'0': 1}


True
